# MediMate-AI: Clinical Risk & Triage Model Training

This notebook documents the data preprocessing, exploratory data analysis, and machine learning model training for the **MediMate-AI** clinical triage classifier.

### Objectives:
1. Ingest physiological telemetry parameters (Blood Pressure, Heart Rate, Glucose, Age).
2. Train a multi-class Random Forest classifier to predict clinical risk level:
   - **0**: Normal / Stable
   - **1**: Moderate Risk
   - **2**: High Risk / Urgent Alert
3. Evaluate model performance and export model weights for the Preethi AI clinical engine.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib

## 1. Dataset Generation and Ingestion

In [2]:
np.random.seed(42)
n_samples = 1000

age = np.random.randint(18, 85, n_samples)
systolic = np.random.normal(125, 20, n_samples).clip(85, 210)
diastolic = np.random.normal(82, 12, n_samples).clip(55, 130)
heart_rate = np.random.normal(76, 14, n_samples).clip(45, 160)
glucose = np.random.normal(105, 30, n_samples).clip(65, 350)

# Determine risk targets based on AHA clinical thresholds
risk = np.zeros(n_samples, dtype=int)
for i in range(n_samples):
    if systolic[i] >= 140 or diastolic[i] >= 90 or glucose[i] >= 140:
        risk[i] = 1
    if systolic[i] >= 160 or diastolic[i] >= 105 or glucose[i] >= 200 or heart_rate[i] >= 120:
        risk[i] = 2

df = pd.DataFrame({
    'age': age,
    'systolic': systolic,
    'diastolic': diastolic,
    'heart_rate': heart_rate,
    'glucose': glucose,
    'risk_level': risk
})

df.head()

## 2. Train / Test Split and Classifier Fitting

In [3]:
features = ['age', 'systolic', 'diastolic', 'heart_rate', 'glucose']
X = df[features]
y = df['risk_level']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Validation Accuracy: {acc * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

## 3. Model Serialization

In [4]:
model_dir = '../saved_models'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'triage_model.pkl')
joblib.dump(rf_model, model_path)
print(f"Trained model weights serialized to: {model_path}")